# **Aplicaçao KNN Alocamento de investimentos**

O algoritmo implementa uma abordagem de Filtragem Colaborativa baseada em Utilizadores (User-Based Collaborative Filtering)


As principais funções do código são :

* **Comparação entre Utilizadores:** O modelo k-NN (k-Nearest Neighbors) utiliza métricas de distância (distância Euclidiana e similaridade por Cosseno) para medir a proximidade entre os vetores de alocação dos diferentes investidores (utilizadores).   

* **Recomendação por Vizinhaça:** Para sugerir uma carteira a um determinado investidor (ou a um novo investidor), o algoritmo identifica os 3 investidores mais semelhantes ($k=3$) e calcula a média das suas alocações de ativos (np.mean).  

* **Diferença para Filtragem Baseada em Conteúdo:** O código não analisa os metadados ou propriedades intrínsecas dos ativos financeiros (como risco, volatilidade, rentabilidade ou setor das ações/fundos).

*  A recomendação é gerada exclusivamente a partir dos padrões de comportamento e alocação de investidores com perfis pares.

In [ ]:
import pandas as pd
import numpy as np

# Definir semente aleatória para reproduzibilidade
np.random.seed(42)

# Parâmetros
n_investidores = 50
investimentos = ['Ações', 'Renda Fixa', 'Imóveis', 'Criptomoedas', 'Fundos Multimercado']

# Gerar valores aleatórios para os 50 investidores
# Usamos a distribuição de Dirichlet para garantir que a soma da linha seja sempre 1 (100%)
# O parâmetro alpha controla o equilíbrio/variabilidade dos aportes
dados_porcentagem = np.random.dirichlet(alpha=np.ones(len(investimentos)), size=n_investidores)

# Multiplicar por 100 para exibir em formato de porcentagem de 0 a 100%
dados_porcentagem_pct = np.round(dados_porcentagem * 100, 2)

# Criar os nomes/IDs dos investidores para os índices das linhas
indices_investidores = [f'Investidor_{i+1:02d}' for i in range(n_investidores)]

# Criar o DataFrame do Pandas
df_carteiras = pd.DataFrame(
    data=dados_porcentagem_pct,
    index=indices_investidores,
    columns=investimentos
)


--- Primeiras 10 linhas do Dataset de Investidores ---
               Ações  Renda Fixa  Imóveis  Criptomoedas  Fundos Multimercado
Investidor_01   7.98       51.20    22.40         15.53                 2.89
Investidor_02   3.86        1.36    45.80         20.93                28.04
Investidor_03   0.36       60.93    31.07          4.15                 3.49
Investidor_04   9.13       16.35    33.52         25.49                15.51
Investidor_05  37.74        5.99    13.78         18.20                24.29
Investidor_06  44.87        6.50    21.06         26.18                 1.39
Investidor_07  12.42        2.48     0.89         39.47                44.74
Investidor_08  42.90        9.43     2.67         29.93                15.06
Investidor_09   3.67       19.26     0.99         67.64                 8.44
Investidor_10  34.06       11.71    23.02         24.81                 6.41

Verificação das somas das linhas (deve ser 100% para cada investidor):
Investidor_01    100.00
In

In [ ]:

# Exibir as 10 primeiras linhas do dataset
print("--- Primeiras 10 linhas do Dataset de Investidores ---")
print(df_carteiras.head(10))


--- Primeiras 10 linhas do Dataset de Investidores ---
               Ações  Renda Fixa  Imóveis  Criptomoedas  Fundos Multimercado
Investidor_01   7.98       51.20    22.40         15.53                 2.89
Investidor_02   3.86        1.36    45.80         20.93                28.04
Investidor_03   0.36       60.93    31.07          4.15                 3.49
Investidor_04   9.13       16.35    33.52         25.49                15.51
Investidor_05  37.74        5.99    13.78         18.20                24.29
Investidor_06  44.87        6.50    21.06         26.18                 1.39
Investidor_07  12.42        2.48     0.89         39.47                44.74
Investidor_08  42.90        9.43     2.67         29.93                15.06
Investidor_09   3.67       19.26     0.99         67.64                 8.44
Investidor_10  34.06       11.71    23.02         24.81                 6.41


In [ ]:

# Validação: Confirmar que a soma de cada linha é 100%
print("\nVerificação das somas das linhas (deve ser 100% para cada investidor):")
print(df_carteiras.sum(axis=1).head(5))


Verificação das somas das linhas (deve ser 100% para cada investidor):
Investidor_01    100.00
Investidor_02     99.99
Investidor_03    100.00
Investidor_04    100.00
Investidor_05    100.00
dtype: float64


In [ ]:
# Transformar o DataFrame em uma matriz NumPy
matriz_investimentos = df_carteiras.to_numpy()

# Exibir a matriz e suas propriedades
print("--- Matriz de Investimentos (NumPy) ---")
print(matriz_investimentos[:5])  # Exibe os 5 primeiros investidores (linhas)

print("\nFormato da matriz (Linhas, Colunas):", matriz_investimentos.shape)
print("Tipo de dado:", type(matriz_investimentos))

--- Matriz de Investimentos (NumPy) ---
[[ 7.98 51.2  22.4  15.53  2.89]
 [ 3.86  1.36 45.8  20.93 28.04]
 [ 0.36 60.93 31.07  4.15  3.49]
 [ 9.13 16.35 33.52 25.49 15.51]
 [37.74  5.99 13.78 18.2  24.29]]

Formato da matriz (Linhas, Colunas): (50, 5)
Tipo de dado: <class 'numpy.ndarray'>


In [ ]:
from sklearn.neighbors import NearestNeighbors

## **KNN_media Euclidiana**

In [ ]:
knn = NearestNeighbors(n_neighbors=3,metric='euclidean')
knn.fit(matriz_investimentos)

NearestNeighbors(metric='euclidean', n_neighbors=3)

In [ ]:
for i in range(len(matriz_investimentos)):
    # Seleciona o vetor de alocaçãodo investido i
    investidor_atual = matriz_investimentos[i]

    #Encontra os vizinhos mais próximos do investido  i, e retorna as distancias
    distancias,indices = knn.kneighbors([investidor_atual])

    #indice[0] é a lista de vizinhos para o investidor i
    #retira ele mesmo da análise
    investidores_vizinhos = indices[0][1:]


    #Pega as alocações dos vizinhos na matriz original
    alocacao_vizinhos = matriz_investimentos[investidores_vizinhos]


    alocacao_media = np.mean(alocacao_vizinhos,axis=0)

    print(f'Recomendação para investidor {i+1}:{alocacao_media}')

Recomendação para investidor 0+1:[11.855 48.86  24.305 10.24   4.74 ]
Recomendação para investidor 1+1:[ 5.655  5.465 36.565 24.7   27.615]
Recomendação para investidor 2+1:[ 4.905 48.47  28.185  9.05   9.39 ]
Recomendação para investidor 3+1:[13.935 10.78  29.16  28.185 17.94 ]
Recomendação para investidor 4+1:[39.365  4.98  12.695 19.03  23.93 ]
Recomendação para investidor 5+1:[36.64  14.505 22.16  20.805  5.895]
Recomendação para investidor 6+1:[27.385  3.37   2.015 43.63  23.605]
Recomendação para investidor 7+1:[38.345  8.14   0.49  36.355 16.67 ]
Recomendação para investidor 8+1:[13.395 13.3    7.09  54.33  11.89 ]
Recomendação para investidor 9+1:[35.555 15.46  23.455 18.68   6.85 ]
Recomendação para investidor 10+1:[36.64  14.505 22.16  20.805  5.895]
Recomendação para investidor 11+1:[60.195  8.735 10.035  4.24  16.795]
Recomendação para investidor 12+1:[ 8.525 11.815 50.255 19.23  10.17 ]
Recomendação para investidor 13+1:[ 9.51   8.75  11.07   8.465 62.2  ]
Recomendação par

## **Similaridade por cosseno**

In [ ]:
knn = NearestNeighbors(n_neighbors=3,metric='cosine')
knn.fit(matriz_investimentos)

NearestNeighbors(metric='cosine', n_neighbors=3)

In [ ]:
for i in range(len(matriz_investimentos)):
    # Seleciona o vetor de alocaçãodo investido i
    investidor_atual = matriz_investimentos[i]

    #Encontra os vizinhos mais próximos do investido  i, e retorna as distancias
    distancias,indices = knn.kneighbors([investidor_atual])

    #indice[0] é a lista de vizinhos para o investidor i
    #retira ele mesmo da análise
    investidores_vizinhos = indices[0][1:]


    #Pega as alocações dos vizinhos na matriz original
    alocacao_vizinhos = matriz_investimentos[investidores_vizinhos]


    alocacao_media = np.mean(alocacao_vizinhos,axis=0)

    print(f'Recomendação para investidor {i+1}:{alocacao_media}')

Recomendação para investidor 1:[11.855 48.86  24.305 10.24   4.74 ]
Recomendação para investidor 2:[ 5.655  5.465 36.565 24.7   27.615]
Recomendação para investidor 3:[ 4.905 48.47  28.185  9.05   9.39 ]
Recomendação para investidor 4:[ 5.725  9.925 37.56  28.34  18.445]
Recomendação para investidor 5:[39.365  4.98  12.695 19.03  23.93 ]
Recomendação para investidor 6:[36.64  14.505 22.16  20.805  5.895]
Recomendação para investidor 7:[27.385  3.37   2.015 43.63  23.605]
Recomendação para investidor 8:[38.345  8.14   0.49  36.355 16.67 ]
Recomendação para investidor 9:[ 5.56  24.135 14.3   48.68   7.325]
Recomendação para investidor 10:[38.38  10.06  23.335 23.37   4.855]
Recomendação para investidor 11:[36.64  14.505 22.16  20.805  5.895]
Recomendação para investidor 12:[60.195  8.735 10.035  4.24  16.795]
Recomendação para investidor 13:[ 5.925 14.165 66.775  5.25   7.885]
Recomendação para investidor 14:[ 9.51   8.75  11.07   8.465 62.2  ]
Recomendação para investidor 15:[32.835  5.

## **Novo investidor**

In [ ]:
novo_investidor = [31.45, 18.22,7.89,24.10,18.34]

In [ ]:
distancias, indices = knn.kneighbors([novo_investidor])

In [ ]:
alocacao_vizinhos = matriz_investimentos[indices[0]]


In [ ]:
alocacao_media = np.mean(alocacao_vizinhos, axis=0)

In [ ]:
print(f'Recomendação para novo investidor {alocacao_media}')

Recomendação para novo investidor [37.09 21.1   3.07 26.27 12.47]
